# Image Generation Infrastructure

**Module:** 17 — Image Generation

Serving, queues, caching, GPU autoscaling, safety services, and observability.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Sketch a reference architecture for image generation at scale
- Apply batching, caching, distillation, resolution ladders
- Define SLOs, tracing fields, and cost controls
- Place safety and provenance on the critical path


## Reference Architecture

### Definition
Separate **API**, **job queue**, **GPU workers**, **model registry**, **safety**, **object storage**, **observability**.

### Why it matters
Sync HTTP to a hot GPU works for demos and fails for spikes and 30s gens.

### How it works
Enqueue → worker → model → safety → object storage → webhook/poll → CDN.

### Intuition
Restaurant: hosts (API), tickets (queue), chefs (GPUs), health inspection (safety).

### Pitfalls
- Blocking web workers on multi-minute jobs
- No model version pinning
- Safety after download

### When to use
Any multi-user product beyond a local ComfyUI box.


```mermaid
flowchart LR
  Client --> API[API / Auth]
  API --> Q[(Queue)]
  Q --> W[GPU Workers]
  W --> REG[Model Registry]
  W --> SAF[Safety]
  SAF --> S3[(Object Storage)]
  S3 --> CDN[CDN]
  W --> OBS[Traces / Metrics]
```

| Concern | Practice |
|---------|----------|
| Authz | Per-tenant quotas, private assets |
| Idempotency | Job keys for retries |
| Backpressure | Queue depth alerts |
| Provenance | Prompt, model hash, seed, C2PA |
| PII | Redact EXIF; signed URLs |


In [ ]:
# Demo 1: in-memory job queue
from collections import deque
from dataclasses import dataclass

@dataclass
class Job:
    id: str
    prompt: str
    status: str = "queued"
    result_uri: str | None = None

class ImageQueue:
    def __init__(self):
        self.q = deque(); self.jobs = {}
    def enqueue(self, job: Job):
        self.jobs[job.id] = job; self.q.append(job.id); return job.id
    def run_one(self, infer):
        jid = self.q.popleft(); job = self.jobs[jid]
        job.status = "running"; job.result_uri = infer(job.prompt); job.status = "done"
        return job

q = ImageQueue(); q.enqueue(Job("j1", "red cube"))
print(q.run_one(lambda p: f"s3://bucket/{hash(p) & 0xffff:x}.png"))


In [ ]:
# Demo 2: cache key
import json, hashlib

def cache_key(model, prompt, seed, w, h, params):
    raw = json.dumps({"m": model, "p": prompt, "s": seed, "w": w, "h": h, "params": params}, sort_keys=True)
    return hashlib.sha256(raw.encode()).hexdigest()

k1 = cache_key("sdxl", "cat", 1, 1024, 1024, {"cfg": 7, "steps": 30})
k2 = cache_key("sdxl", "cat", 1, 1024, 1024, {"steps": 30, "cfg": 7})
k3 = cache_key("sdxl", "cat", 2, 1024, 1024, {"cfg": 7, "steps": 30})
print(k1 == k2, k1 == k3)


## Performance Tactics

### Definition
Cut latency/cost with distilled models, caches, resolution ladders, dynamic batching, multi-tenant GPU packing, speculative previews.

### Why it matters
Unit economics decide whether the feature ships.

### How it works
Measure p50/p95; optimize hottest stage; offer turbo vs quality tiers.

### Intuition
Express lanes + carpool lanes for GPUs.

### Pitfalls
- Batching incompatible resolutions
- Caching without safety re-check after policy change

### When to use
High QPS previews, burst campaigns, free tiers.


| Tactic | Saves | Risk |
|--------|-------|------|
| Distilled 4–8 step | Latency | Different look |
| Draft 512 then refine | Cost | Drift |
| Result cache | $ | Stale after policy/model change |
| Dynamic batching | Throughput | Tail latency spikes |
| Quantization | VRAM | Quality cliffs |


In [ ]:
# Demo 3: workers needed for SLA
import math

def workers_needed(qps, seconds_per_job, target_util=0.7):
    return math.ceil((qps * seconds_per_job) / target_util)

for qps in [0.5, 2, 10]:
    print("qps", qps, "workers", workers_needed(qps, 6.0))


In [ ]:
# Demo 4: trace fields
import json
trace = {"trace_id": "tr_123", "tenant": "acme", "model": "flux-dev@sha256:abc", "seed": 42,
         "queue_ms": 120, "infer_ms": 5400, "safety_ms": 80, "status": "ok", "safety_labels": []}
print(json.dumps(trace, indent=2))


### Try it yourself — Infra

1. Add webhook completion + signed URL expiry to the queue demo.
2. Define SLOs: availability, p95 latency, safety proxies.
3. Design a model registry record: name, hash, VAE, sampler, license.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `job queue` | Buffer between API and GPU workers |
| `model registry` | Versioned model metadata/artifacts |
| `dynamic batching` | Group concurrent requests for GPU efficiency |
| `resolution ladder` | Cheap preview then optional refine |


### Workshop — Parameter journal — Image Infra

List every knob you touched. Predict the effect of changing one knob before changing it.


In [ ]:
# Workshop 1 — Image Infra
knobs = ['seed','guidance','steps','size','strength']
for k in knobs:
    print(f'{k}: value=?, hypothesis=?, observed=?')


### Workshop — Failure taxonomy — Image Infra

Classify bad outputs into: prompt, model, control, safety, or infra.


In [ ]:
# Workshop 2 — Image Infra
examples = ['ignored count','flicker','pose ignored','blocked','504']
for e in examples:
    print(e, '->', 'TODO-label')


### Workshop — Cost / latency card — Image Infra

Estimate unit cost for draft vs final tiers at a daily volume.


In [ ]:
# Workshop 3 — Image Infra
def monthly(qpd, price, days=30, retry=0.1):
    return round(qpd*days*(1+retry)*price, 2)
print('draft$', monthly(2000, 0.02))
print('final$', monthly(500, 0.08))


### Workshop — Eval golden item — Image Infra

Add one golden prompt/job with must-have attributes and reject criteria.


In [ ]:
# Workshop 4 — Image Infra
golden = {'id':'g1','prompt':'TODO','must_have':['subject','style'],'reject_if':['watermark']}
print(golden)


### Workshop — Safety + provenance — Image Infra

Write audit metadata: model hash, seed, policy version, credentials flag.


In [ ]:
# Workshop 5 — Image Infra
meta = {'model':'name@sha256:...','seed':0,'policy_version':'2026.04','credentials':True}
assert 'model' in meta
print(meta)


### Workshop — Ablation plan — Image Infra

Design a 4-run ablation changing only one variable each time.


In [ ]:
# Workshop 6 — Image Infra
runs = [{'id':i,'change':c,'score':None} for i,c in enumerate(['baseline','guidance-2','steps+10','new_seed'],1)]
print(runs)


### Workshop — Interface sketch — Image Infra

Sketch provider-agnostic request/response dicts for this modality.


In [ ]:
# Workshop 7 — Image Infra
req = {'prompt':'...','size':'...','seed':1}
resp = {'status':'ok','asset_uri':'...','model':'...','seed':1}
print(req); print(resp)


### Workshop — Self-check — Image Infra

Run this checklist printer and fill it honestly before moving on.


In [ ]:
# Workshop 8 — Image Infra
for i,x in enumerate(['restated objectives','ran demos','named pitfall','named metric'],1):
    print(f'{i}. [ ] {x}')


## Key Takeaways

- Async jobs + storage + safety on the critical path
- Pin model hashes; cache carefully
- Optimize with measured SLOs and tiered quality
